# PETR4.SA — LSTM Stock Price Predictor
**Tech Challenge Fase 4 | POSTECH MLET**

Pipeline completa: coleta → EDA → pré-processamento → treino → avaliação.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib, os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

plt.style.use('seaborn-v0_8-darkgrid')
print('TensorFlow:', tf.__version__)

## 1. Coleta de Dados

In [ ]:
TICKER = 'PETR4.SA'
START  = '2018-01-01'
END    = '2024-12-31'

df = yf.download(TICKER, start=START, end=END, auto_adjust=True)
print(f'Shape: {df.shape}')
df.head()

## 2. Análise Exploratória (EDA)

In [ ]:
df.info()
print('\nNulos:', df.isnull().sum().sum())
df.describe()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(df['Close'], linewidth=1)
axes[0].set_title('PETR4.SA — Preço de Fechamento')
axes[0].set_ylabel('Preço (R$)')

axes[1].bar(df.index, df['Volume'], width=1, alpha=0.5, color='steelblue')
axes[1].set_title('Volume Negociado')
axes[1].set_ylabel('Volume')

plt.tight_layout()
plt.show()

In [ ]:
# Média móvel 20 e 50 dias
df['MA20'] = df['Close'].rolling(20).mean()
df['MA50'] = df['Close'].rolling(50).mean()

plt.figure(figsize=(14, 5))
plt.plot(df['Close'], label='Fechamento', linewidth=1)
plt.plot(df['MA20'],  label='MM 20d', linewidth=1.5)
plt.plot(df['MA50'],  label='MM 50d', linewidth=1.5)
plt.title('PETR4.SA — Médias Móveis')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Pré-processamento

In [ ]:
SEQ_LEN     = 60
TRAIN_RATIO = 0.8

close = df[['Close']].values

scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(close)

split      = int(len(scaled) * TRAIN_RATIO)
train_data = scaled[:split]
test_data  = scaled[split - SEQ_LEN:]

def build_sequences(data, seq_len):
    X, y = [], []
    for i in range(seq_len, len(data)):
        X.append(data[i-seq_len:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = build_sequences(train_data, SEQ_LEN)
X_test,  y_test  = build_sequences(test_data,  SEQ_LEN)

X_train = X_train.reshape(-1, SEQ_LEN, 1)
X_test  = X_test.reshape(-1,  SEQ_LEN, 1)

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

## 4. Construção e Treinamento do Modelo LSTM

In [ ]:
model = Sequential([
    LSTM(128, return_sequences=True, input_shape=(SEQ_LEN, 1)),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Loss durante o Treinamento')
plt.xlabel('Épocas')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Avaliação do Modelo

In [ ]:
y_pred = model.predict(X_test)

y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred_inv = scaler.inverse_transform(y_pred).flatten()

mae  = mean_absolute_error(y_test_inv, y_pred_inv)
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv)) * 100

print(f'MAE  : R$ {mae:.4f}')
print(f'RMSE : R$ {rmse:.4f}')
print(f'MAPE : {mape:.2f}%')

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(y_test_inv, label='Real',     linewidth=1.5)
plt.plot(y_pred_inv, label='Previsto', linewidth=1.5, linestyle='--')
plt.title('PETR4.SA — Real vs Previsto (conjunto de teste)')
plt.xlabel('Dias')
plt.ylabel('Preço de Fechamento (R$)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Salvamento do Modelo

In [ ]:
model.save('../model/lstm_model.keras')
joblib.dump(scaler, '../model/scaler.pkl')
print('Modelo e scaler salvos!')